# 04 — Guardrails y confirmación humana

**Bloque 3, 20 minutos.** El código vive en
[`src/paso4_guardrails/agente.py`](../src/paso4_guardrails/agente.py).

El paso 3 dejó un agente que funciona. Este lo deja en condiciones de que
alguien lo conecte a algo que importe.

Y arrastraba un cabo suelto: `actualizar_existencias` **modifica el sistema** y
nadie le pidió permiso a nadie. En un inventario de prueba no pasa nada. Este
paso agrega una tool que **borra sin papelera**, y ahí sí.

Corre el notebook de arriba abajo; tarda unos cuatro minutos. Una de las celdas
borra un producto de verdad, y la última lo deja todo como estaba.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")
import paso4_guardrails.agente as p4
importlib.reload(p4)

import requests
print("backend:", p4.BACKEND)
print("tools  :", list(p4.DISPONIBLES))
print("destructivas:", p4.DESTRUCTIVAS)

def tabla():
    d = requests.get(f"{p4.BACKEND}/api/inventario", timeout=10).json()
    for p in d["productos"]:
        print(f"  {p['sku']:<15} {p['nombre']:<30} {p['existencias']:>4}")
    print(f"  {d['total']} productos")

def reiniciar():
    requests.post(f"{p4.BACKEND}/api/admin/reiniciar", timeout=20)
    print("inventario reiniciado")

## Aquí el bucle sí cambia

En los pasos 2 y 3, `correr()` era **idéntico carácter por carácter**. Ese era
el punto del paso 3: cambió lo que había debajo de las tools, no el bucle.

Ahora crece. La celda siguiente mide cuánto. Lo que importa es que **todo lo que
creció hace lo mismo**: decidir si una petición del modelo se ejecuta o no.

In [ ]:
import inspect
import paso3_tools_http.agente as p3

a = inspect.getsource(p3.correr).split("\n")
b = inspect.getsource(p4.correr).split("\n")
print(f"  bucle del paso 3: {len(a)} líneas")
print(f"  bucle del paso 4: {len(b)} líneas   (+{len(b) - len(a)})")

## Los cinco guardrails

Dos ya estaban desde el paso 2 y nunca se llamaron por su nombre.

| # | Guardrail | Contra qué | ¿Nuevo? |
|---|---|---|---|
| 1 | **Lista blanca** | tools inventadas | ya estaba: el diccionario `DISPONIBLES` |
| 2 | **Tope de vueltas** | el bucle infinito | ya estaba: `MAX_VUELTAS` |
| 3 | **Antirrepetición** | repetir lo mismo sin avanzar | nuevo |
| 4 | **Validación de argumentos** | basura antes de que llegue al sistema | nuevo |
| 5 | **Confirmación humana** | lo que no se puede deshacer | nuevo |

Antes de seguir, vale responder una pregunta: **¿cuáles de estos cinco son "de
IA"?**

Ninguno. Son lista blanca, límite de recursos, validación de entrada y control
de acceso, las mismas cuatro cosas de siempre. **Lo único nuevo es que la
entrada en la que no se puede confiar ahora la genera el propio sistema.**

## Guardrail 4: validar antes de ejecutar

El backend ya valida. El agente valida otra vez, antes, y a propósito.

`validar()` es una función común que devuelve `None` si está bien o un texto si
no. La celda siguiente la prueba sola, sin agente de por medio.

In [ ]:
CASOS = [
    ("actualizar_existencias", {"sku": "CAFE-500", "existencias": 17}),
    ("actualizar_existencias", {"sku": "CAFE-500", "existencias": -5}),
    ("actualizar_existencias", {"sku": "CAFE-500", "existencias": 99999}),
    ("actualizar_existencias", {"sku": "CAFE-500", "existencias": "muchas"}),
    ("consultar_stock", {"sku": "cafe molido 500g"}),
    ("consultar_stock", {"sku": "CAFE-500"}),
]
for nombre, args in CASOS:
    falla = p4.validar(nombre, args)
    print(f"  {'PASA ' if falla is None else 'PARA '}  {str(args)[:46]:<48} {falla or ''}")

### Por qué validar dos veces

- **El error llega más claro y más barato.** Sin viaje de red, y redactado por ti
  en lugar de por el framework del backend.
- **Hay reglas que el backend no conoce.** "No borres más de uno por petición" o
  "no toques esta categoría" son reglas **del agente**, no del dominio.
- **El backend es un consumidor más.** Si mañana el agente llama a otra cosa, la
  regla viaja con el agente.

La celda siguiente la muestra actuando dentro del bucle.

In [ ]:
reiniciar()
p4.correr("Pon el stock de CAFE-500 en 99999 unidades", confirmar=p4.rechazar_todo)

Fíjate en la línea `[BLOQUEADO por validacion]`: la tool **no se ejecutó**. Y el
modelo, al recibir el motivo, se lo explicó a la persona y ofreció una
alternativa en lugar de insistir.

> El tope de 10 000 unidades es **arbitrario**, y por eso vale discutirlo: ese
> número no sale de conocer el modelo, sale de conocer el negocio. Es el
> `TODO 3` del archivo.

## Guardrail 5: la confirmación humana

Ahora la tool que de verdad duele: `borrar_producto`. **No hay papelera.**

Que duela es el punto. Un guardrail sobre algo reversible no enseña nada: si
deshacer es gratis, confirmar sobra.

`correr()` recibe un parámetro `confirmar`. Por omisión se detiene y pregunta
con `input()`. Las celdas siguientes usan dos atajos para no depender de que
alguien teclee:

| | |
|---|---|
| `p4.rechazar_todo` | dice que no sin preguntar |
| `p4.aprobar_todo` | dice que sí sin preguntar, o sea **el guardrail apagado** |

In [ ]:
reiniciar()
print("\nTE-200 antes:", requests.get(f"{p4.BACKEND}/api/inventario/TE-200", timeout=10).status_code)

p4.correr("Borra el producto TE-200 del inventario", confirmar=p4.rechazar_todo)

print("\nTE-200 después:", requests.get(f"{p4.BACKEND}/api/inventario/TE-200", timeout=10).status_code,
      "  (200 = sigue vivo)")

Dos cosas pasaron ahí:

**El rechazo le vuelve al modelo como resultado de la tool**, no como una
excepción. Por eso puede explicárselo a la persona —*"no puedo borrarlo porque
no se autorizó"*— en lugar de que el programa se caiga.

**El producto sigue existiendo.** El guardrail no avisó: detuvo.

### Y ahora con el guardrail apagado

La celda siguiente hace la misma petición con `aprobar_todo`. Es la demostración
de falla número 4 del bloque 5, y **borra un producto de verdad**. La última
celda del notebook deja el inventario como estaba.

In [ ]:
reiniciar()
p4.correr("Borra el producto TE-200 del inventario", confirmar=p4.aprobar_todo)

r = requests.get(f"{p4.BACKEND}/api/inventario/TE-200", timeout=10)
print(f"\nTE-200 después: HTTP {r.status_code}   (404 = ya no existe)")
print("\nY el inventario quedó así:")
tabla()

El producto se borró. Sin preguntar, sin papelera, en una sola vuelta.

Fíjate en cuál borró: `TE-200`, té verde, que tenía **0 existencias**. Era el
candidato más plausible para "ya no lo vendemos". El modelo eligió
razonablemente y aun así **no era su decisión**.

Esa es la pregunta de diseño del paso 4, y no tiene respuesta única:

> **¿Qué acciones merecen una pausa?**

Si se confirman todas, nadie las lee y la confirmación se vuelve un clic
reflejo. Si no se confirma ninguna, el agente borra algo un martes.

## Guardrail 3: antirrepetición

El tope de vueltas corta el bucle infinito, pero **tarde y sin explicar por
qué**: el agente gasta ocho llamadas al modelo y termina con "me quedé sin
vueltas".

La repetición es la forma más común de ese bucle: pedir exactamente lo mismo
esperando otro resultado. Detectarla es barato —guardar el par `(nombre,
argumentos)` de lo ya ejecutado— y el mensaje que le devuelves es mucho más
útil que el silencio.

In [ ]:
# Lo probamos directo, sin depender de que el modelo entre en bucle solo.
import json as _json

vistas = []
for i in range(3):
    firma = ("consultar_stock", _json.dumps({"sku": "CAFE-500"}, sort_keys=True))
    if firma in vistas:
        print(f"  intento {i+1}: BLOQUEADO — ya se ejecutó, el resultado sería el mismo")
    else:
        vistas.append(firma)
        print(f"  intento {i+1}: ejecuta")

## Lo que no resuelve ningún guardrail

Vuelve un momento al notebook 02, a la celda donde el modelo inventaba cinco
cantidades. **Ningún guardrail de esta lista lo habría detenido:**

- La tool existía, así que pasa la lista blanca.
- Los argumentos eran cadenas razonables, así que pasa la validación.
- No repetía, así que pasa la antirrepetición.
- `consultar_stock` no es destructiva, así que no pide confirmación.

Los guardrails controlan **lo que el agente hace**. No controlan **lo que el
agente dice**. Para eso hacen falta otras dos cosas, las dos en el bloque 5:

- **Evaluación**: casos de prueba de entrada contra efecto esperado.
- **Trazabilidad**: que la respuesta cite de qué llamada salió cada número.

Que un agente tenga guardrails no quiere decir que sea confiable. Quiere decir
que lo que rompa, lo rompe dentro de los límites que tú pusiste.

## Tu turno

Los cuatro `# TODO (taller)` del final de
[`src/paso4_guardrails/agente.py`](../src/paso4_guardrails/agente.py).

El `TODO 1` es el más instructivo: agrega `actualizar_existencias` a
`DESTRUCTIVAS` y vuelve a correr las celdas del paso 3. Funciona, pero cuenta
cuántas confirmaciones de más aparecen y pregúntate si al décimo clic seguirías
leyendo lo que te preguntan.

In [ ]:
reiniciar()
tabla()